# SE3082 - Advanced Topics in Software Engineering
## Lab 10: CUDA Timing, Error Checking, 2D and 3D Kernels

**Student ID:** IT24610790  
**Student Name:** Akini Panapitiya  


In [ ]:
!nvidia-smi
!nvidia-smi --query-gpu=name,compute_cap --format=csv

Thu Oct  8 01:41:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!nvcc --version


nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Aug_20_01:58:59_PM_PDT_2025
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0


In [ ]:
!nvidia-smi --query-gpu=name,compute_cap --format=csv

name, compute_cap
Tesla T4, 7.5


Exercise 1

In [ ]:
%%writefile ex1_timed.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>
#define N 512

__global__ void add(int *a, int *b, int *c) {
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    int size = N * sizeof(int);
    double dStart, dElaps;

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = (int *)malloc(size); random_ints(a, N);
    b = (int *)malloc(size); random_ints(b, N);
    c = (int *)malloc(size);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    dStart = cpuSecond();
    add<<<N, 1>>>(d_a, d_b, d_c);
    cudaDeviceSynchronize();
    dElaps = cpuSecond() - dStart;

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    for (int r = 0; r < 5; r++)
        printf("%d + %d = %d\n", a[r], b[r], c[r]);

    int errors = 0;
    for (int r = 0; r < N; r++)
        if (c[r] != a[r] + b[r]) errors++;
    printf("Verification: %d mismatches out of %d\n", errors, N);
    printf("Time taken - %.7f\n", dElaps);

    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}

Overwriting ex1_timed.cu


In [ ]:
!nvcc -arch=native -o ex1_timed.o ex1_timed.cu && ./ex1_timed.o

83 + 77 = 160
86 + 34 = 120
77 + 90 = 167
15 + 26 = 41
93 + 24 = 117
Verification: 0 mismatches out of 512
Time taken - 0.0000899


Removing cudaDeviceSynchronize() dropped the time from 0.0006182s to 0.0000980s - roughly 6x “faster.”

Explanation: add<<<N,1>>>(...) is an asynchronous kernel launch, the CPU issues it and immediately moves to the next line without waiting for the GPU to finish. Without cudaDeviceSynchronize(), cpuSecond() is called right after the launch call returns (which takes almost no time), so you’re only timing how long it takes to queue the kernel, not how long it takes to run. The cudaMemcpy that follows happens to be synchronizing (it blocks until the kernel completes before copying results back), which is why the output is still correct but the timer had already stopped before that wait happened, so the measured time is meaningless as a performance number. cudaDeviceSynchronize() is required to get an accurate kernel execution time.

Lab 09-Exercise 3

In [ ]:
%%writefile ex3_blocks_timed.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>
#define N 512

__global__ void add(int *a, int *b, int *c) {
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++) x[i] = rand() % 100;
}

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c, *d_a, *d_b, *d_c;
    int size = N * sizeof(int);
    double dStart, dElaps;

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = (int *)malloc(size); random_ints(a, N);
    b = (int *)malloc(size); random_ints(b, N);
    c = (int *)malloc(size);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    dStart = cpuSecond();
    add<<<N, 1>>>(d_a, d_b, d_c);
    cudaDeviceSynchronize();
    dElaps = cpuSecond() - dStart;

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    int errors = 0;
    for (int r = 0; r < N; r++) if (c[r] != a[r] + b[r]) errors++;
    printf("Verification: %d mismatches out of %d\n", errors, N);
    printf("Time taken - %.7f\n", dElaps);

    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}

Writing ex3_blocks_timed.cu


In [ ]:
!nvcc -arch=native -o ex3_blocks_timed.o ex3_blocks_timed.cu
!./ex3_blocks_timed.o
!./ex3_blocks_timed.o
!./ex3_blocks_timed.o

Verification: 0 mismatches out of 512
Time taken - 0.0001149
Verification: 0 mismatches out of 512
Time taken - 0.0001211
Verification: 0 mismatches out of 512
Time taken - 0.0001230


Lab 09-Exercise 4

In [ ]:
%%writefile ex4_threads_timed.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>
#define N 512

__global__ void add(int *a, int *b, int *c) {
    c[threadIdx.x] = a[threadIdx.x] + b[threadIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++) x[i] = rand() % 100;
}

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c, *d_a, *d_b, *d_c;
    int size = N * sizeof(int);
    double dStart, dElaps;

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = (int *)malloc(size); random_ints(a, N);
    b = (int *)malloc(size); random_ints(b, N);
    c = (int *)malloc(size);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    dStart = cpuSecond();
    add<<<1, N>>>(d_a, d_b, d_c);
    cudaDeviceSynchronize();
    dElaps = cpuSecond() - dStart;

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    int errors = 0;
    for (int r = 0; r < N; r++) if (c[r] != a[r] + b[r]) errors++;
    printf("Verification: %d mismatches out of %d\n", errors, N);
    printf("Time taken - %.7f\n", dElaps);

    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}

Writing ex4_threads_timed.cu


In [ ]:
!nvcc -arch=native -o ex4_threads_timed.o ex4_threads_timed.cu
!./ex4_threads_timed.o
!./ex4_threads_timed.o
!./ex4_threads_timed.o

Verification: 0 mismatches out of 512
Time taken - 0.0001161
Verification: 0 mismatches out of 512
Time taken - 0.0000892
Verification: 0 mismatches out of 512
Time taken - 0.0000880


Lab 09-Exercise 5

In [ ]:
%%writefile ex5_vecmul_timed.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>
#define N 10000000
#define THREADS_PER_BLOCK 512

__global__ void vecmul(int *a, int *b, int *c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) c[i] = a[i] * b[i];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++) x[i] = rand() % 100;
}

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c, *d_a, *d_b, *d_c;
    int size = N * sizeof(int);
    double dStart, dElaps;

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = (int *)malloc(size); random_ints(a, N);
    b = (int *)malloc(size); random_ints(b, N);
    c = (int *)malloc(size);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    int blocks = (N + THREADS_PER_BLOCK - 1) / THREADS_PER_BLOCK;

    dStart = cpuSecond();
    vecmul<<<blocks, THREADS_PER_BLOCK>>>(d_a, d_b, d_c, N);
    cudaDeviceSynchronize();
    dElaps = cpuSecond() - dStart;

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    long errors = 0;
    for (int r = 0; r < N; r++) if (c[r] != a[r] * b[r]) errors++;
    printf("Verification: %ld mismatches out of %d\n", errors, N);
    printf("Time taken - %.7f\n", dElaps);

    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}

Writing ex5_vecmul_timed.cu


In [ ]:
!nvcc -arch=native -o ex5_vecmul_timed.o ex5_vecmul_timed.cu
!./ex5_vecmul_timed.o
!./ex5_vecmul_timed.o
!./ex5_vecmul_timed.o

Verification: 0 mismatches out of 10000000
Time taken - 0.0006042
Verification: 0 mismatches out of 10000000
Time taken - 0.0005960
Verification: 0 mismatches out of 10000000
Time taken - 0.0006070


Lab 09-Exercise 6

In [ ]:
%%writefile ex6_matrix_timed.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>
#define ROWS 10000
#define COLS 10000

__global__ void matmul(int *a, int *b, int *c, int rows, int cols) {
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    if (row < rows && col < cols) {
        int idx = row * cols + col;
        c[idx] = a[idx] * b[idx];
    }
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++) x[i] = rand() % 100;
}

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int n = ROWS * COLS;
    size_t size = (size_t)n * sizeof(int);
    int *a, *b, *c, *d_a, *d_b, *d_c;
    double dStart, dElaps;

    a = (int *)malloc(size); random_ints(a, n);
    b = (int *)malloc(size); random_ints(b, n);
    c = (int *)malloc(size);

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    dim3 threadsPerBlock(32, 16);
    dim3 blocks((COLS + 31) / 32, (ROWS + 15) / 16);

    dStart = cpuSecond();
    matmul<<<blocks, threadsPerBlock>>>(d_a, d_b, d_c, ROWS, COLS);
    cudaDeviceSynchronize();
    dElaps = cpuSecond() - dStart;

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    long errors = 0;
    for (int r = 0; r < n; r++) if (c[r] != a[r] * b[r]) errors++;
    printf("Verification: %ld mismatches out of %d\n", errors, n);
    printf("Time taken - %.7f\n", dElaps);

    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}

Writing ex6_matrix_timed.cu


In [ ]:
!nvcc -arch=native -o ex6_matrix_timed.o ex6_matrix_timed.cu
!./ex6_matrix_timed.o
!./ex6_matrix_timed.o
!./ex6_matrix_timed.o

Verification: 0 mismatches out of 100000000
Time taken - 0.0049520
Verification: 0 mismatches out of 100000000
Time taken - 0.0049641
Verification: 0 mismatches out of 100000000
Time taken - 0.0049169


Lab 09-Exercise 5 at N=100000000

In [ ]:
%%writefile ex5_vecmul_fulltime.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>
#define N 100000000
#define THREADS_PER_BLOCK 512

__global__ void vecmul(int *a, int *b, int *c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) c[i] = a[i] * b[i];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++) x[i] = rand() % 100;
}

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    double fullStart, fullElaps, kernelStart, kernelElaps;
    fullStart = cpuSecond();

    int *a, *b, *c, *d_a, *d_b, *d_c;
    size_t size = (size_t)N * sizeof(int);

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = (int *)malloc(size); random_ints(a, N);
    b = (int *)malloc(size); random_ints(b, N);
    c = (int *)malloc(size);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    int blocks = (N + THREADS_PER_BLOCK - 1) / THREADS_PER_BLOCK;

    kernelStart = cpuSecond();
    vecmul<<<blocks, THREADS_PER_BLOCK>>>(d_a, d_b, d_c, N);
    cudaDeviceSynchronize();
    kernelElaps = cpuSecond() - kernelStart;

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    fullElaps = cpuSecond() - fullStart;

    long errors = 0;
    for (int r = 0; r < N; r++) if (c[r] != a[r] * b[r]) errors++;
    printf("Verification: %ld mismatches out of %d\n", errors, N);
    printf("Kernel-only time   - %.7f\n", kernelElaps);
    printf("Full-sequence time - %.7f\n", fullElaps);

    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}

Writing ex5_vecmul_fulltime.cu


In [ ]:
!nvcc -arch=native -o ex5_vecmul_fulltime.o ex5_vecmul_fulltime.cu
!./ex5_vecmul_fulltime.o
!./ex5_vecmul_fulltime.o
!./ex5_vecmul_fulltime.o

Verification: 0 mismatches out of 100000000
Kernel-only time   - 0.0047619
Full-sequence time - 4.6794329
Verification: 0 mismatches out of 100000000
Kernel-only time   - 0.0047829
Full-sequence time - 4.9827480
Verification: 0 mismatches out of 100000000
Kernel-only time   - 0.0047500
Full-sequence time - 4.6145880


Exercise 2

In [ ]:
%%writefile ex2_check.cu
#include <cstdio>
#include <cstdlib>
#define N 512

#define CHECK(call)                                                    \
{                                                                      \
    const cudaError_t error = call;                                   \
    if (error != cudaSuccess) {                                       \
        printf("Error: %s:%d, ", __FILE__, __LINE__);                 \
        printf("code:%d, reason: %s\n", error, cudaGetErrorString(error)); \
        exit(1);                                                      \
    }                                                                  \
}

__global__ void add(int *a, int *b, int *c) {
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++) x[i] = rand() % 100;
}

int main(void) {
    int *a, *b, *c, *d_a, *d_b, *d_c;
    int size = N * sizeof(int);

    CHECK(cudaMalloc((void **)&d_a, (size_t)1000000000000));  // 1 TB — will fail
    CHECK(cudaMalloc((void **)&d_b, size));
    CHECK(cudaMalloc((void **)&d_c, size));

    a = (int *)malloc(size); random_ints(a, N);
    b = (int *)malloc(size); random_ints(b, N);
    c = (int *)malloc(size);

    CHECK(cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice));

    add<<<N, 1>>>(d_a, d_b, d_c);
    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());

    CHECK(cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost));

    int errors = 0;
    for (int r = 0; r < N; r++) if (c[r] != a[r] + b[r]) errors++;
    printf("Verification: %d mismatches out of %d\n", errors, N);

    free(a); free(b); free(c);
    CHECK(cudaFree(d_a));
    CHECK(cudaFree(d_b));
    CHECK(cudaFree(d_c));
    return 0;
}

Overwriting ex2_check.cu


In [ ]:
!nvcc -arch=native -o ex2_check.o ex2_check.cu && ./ex2_check.o

Error: ex2_check.cu:27, code:2, reason: out of memory


Exercise 2-Experiment C

In [ ]:
%%writefile ex2_nocheck.cu
#include <cstdio>
#include <cstdlib>
#define N 512

__global__ void add(int *a, int *b, int *c) {
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++) x[i] = rand() % 100;
}

int main(void) {
    int *a, *b, *c, *d_a, *d_b, *d_c;
    int size = N * sizeof(int);

    cudaMalloc((void **)&d_a, (size_t)1000000000000);  // bug: no CHECK, 1TB
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = (int *)malloc(size); random_ints(a, N);
    b = (int *)malloc(size); random_ints(b, N);
    c = (int *)malloc(size);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    add<<<N, 1>>>(d_a, d_b, d_c);
    cudaDeviceSynchronize();

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    int errors = 0;
    for (int r = 0; r < N; r++) if (c[r] != a[r] + b[r]) errors++;
    printf("Verification: %d mismatches out of %d\n", errors, N);

    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}

Writing ex2_nocheck.cu


In [ ]:
!nvcc -arch=native -o ex2_nocheck.o ex2_nocheck.cu && ./ex2_nocheck.o

Verification: 512 mismatches out of 512


Exercise 3

In [1]:
%%writefile ex3_picture.cu
#include <cstdio>
#include <cstdlib>
#define WIDTH  800
#define HEIGHT 600

__global__ void PictureKernel(int *d_in, int *d_out, int width, int height, int offset) {
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    if (row < height && col < width) {
        int idx = row * width + col;
        int val = d_in[idx] + offset;
        d_out[idx] = (val > 255) ? 255 : val;
    }
}

int main(void) {
    int n = WIDTH * HEIGHT;
    size_t size = (size_t)n * sizeof(int);
    int offset = 100;

    int *h_in = (int *)malloc(size);
    int *h_out = (int *)malloc(size);
    for (int i = 0; i < n; i++) h_in[i] = rand() % 256;

    int *d_in, *d_out;
    cudaMalloc((void **)&d_in, size);
    cudaMalloc((void **)&d_out, size);
    cudaMemcpy(d_in, h_in, size, cudaMemcpyHostToDevice);

    dim3 threadsPerBlock(16, 16);
    dim3 blocks((WIDTH + 15) / 16, (HEIGHT + 15) / 16);   // 50 x 38

    PictureKernel<<<blocks, threadsPerBlock>>>(d_in, d_out, WIDTH, HEIGHT, offset);
    cudaDeviceSynchronize();
    cudaMemcpy(h_out, d_out, size, cudaMemcpyDeviceToHost);

    int errors = 0, capped = 0;
    for (int i = 0; i < n; i++) {
        int expected = h_in[i] + offset;
        if (expected > 255) { expected = 255; capped++; }
        if (h_out[i] != expected) errors++;
    }
    printf("Grid: %d x %d blocks\n", blocks.x, blocks.y);
    printf("Verification: %d mismatches out of %d\n", errors, n);
    printf("Pixels capped at 255: %d (%.1f%%)\n", capped, 100.0 * capped / n);

    free(h_in); free(h_out);
    cudaFree(d_in); cudaFree(d_out);
    return 0;
}

Writing ex3_picture.cu


In [2]:
!nvcc -arch=native -o ex3_picture.o ex3_picture.cu && ./ex3_picture.o

Grid: 50 x 38 blocks
Verification: 0 mismatches out of 480000
Pixels capped at 255: 187382 (39.0%)


Exercise 4

In [3]:
%%writefile ex4_volume.cu
#include <cstdio>
#include <cstdlib>
#define NX 500
#define NY 500
#define NZ 500

__global__ void VolumeKernel(int *d_in, int *d_out, int nx, int ny, int nz, int offset) {
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;
    int z = blockIdx.z * blockDim.z + threadIdx.z;
    if (x < nx && y < ny && z < nz) {
        int idx = (z * ny + y) * nx + x;
        int val = d_in[idx] + offset;
        d_out[idx] = (val > 255) ? 255 : val;
    }
}

int main(void) {
    size_t n = (size_t)NX * NY * NZ;
    size_t size = n * sizeof(int);
    int offset = 100;

    int *h_in = (int *)malloc(size);
    int *h_out = (int *)malloc(size);
    for (size_t i = 0; i < n; i++) h_in[i] = rand() % 256;

    int *d_in, *d_out;
    cudaMalloc((void **)&d_in, size);
    cudaMalloc((void **)&d_out, size);
    cudaMemcpy(d_in, h_in, size, cudaMemcpyHostToDevice);

    dim3 threadsPerBlock(8, 8, 8);
    dim3 blocks((NX + 7) / 8, (NY + 7) / 8, (NZ + 7) / 8);   // 63 x 63 x 63

    VolumeKernel<<<blocks, threadsPerBlock>>>(d_in, d_out, NX, NY, NZ, offset);
    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess) printf("Launch error: %s\n", cudaGetErrorString(err));
    cudaDeviceSynchronize();
    cudaMemcpy(h_out, d_out, size, cudaMemcpyDeviceToHost);

    long errors = 0, capped = 0;
    for (size_t i = 0; i < n; i++) {
        int expected = h_in[i] + offset;
        if (expected > 255) { expected = 255; capped++; }
        if (h_out[i] != expected) errors++;
    }
    printf("Grid: %d x %d x %d blocks\n", blocks.x, blocks.y, blocks.z);
    printf("Verification: %ld mismatches out of %zu\n", errors, n);
    printf("Voxels capped at 255: %ld (%.1f%%)\n", capped, 100.0 * capped / n);

    free(h_in); free(h_out);
    cudaFree(d_in); cudaFree(d_out);
    return 0;
}

Writing ex4_volume.cu


In [4]:
!nvcc -arch=native -o ex4_volume.o ex4_volume.cu && ./ex4_volume.o

Grid: 63 x 63 x 63 blocks
Verification: 0 mismatches out of 125000000
Voxels capped at 255: 48834659 (39.1%)
